**Introduccion: Refinamiento de la funcion distancia**


El sistema base utiliza la distancia de Levenshtein, que asume que todos los errores humanos son iguales (sustituir 'a' por 's' cuesta lo mismo que 'a' por 'z').

El objetivo de esta mejora es reflejar la realidad del error humano: los usuarios fallan por cercanía física en el teclado o por dudas ortográficas (fonética). 

Al penalizar menos los errores "lógicos", el autocorrector priorizará candidatos que tengan sentido sobre palabras aleatorias que simplemente están a la misma distancia matemática.

In [ ]:
import re
import os

files = os.listdir('/Users/juancho/Desktop/Sistemas_Inteligentes/Practica3_Autocorrector/libros')
words = {}
for file in files:
    with open(f'libros/{file}', 'r', encoding='UTF-8') as f:
        for linea in f:
            linea = linea.lower()
            linea = re.sub(r'[^\w\s]','', linea)

            for word in linea.split():
                words[word] = words.get(word,0) + 1
                
print(words)
print(len(words))


**Funciones para aplicar una edicion**

Aplicamos las 4 funciones base, insertar, borrar, reemplazar e intercambiar:

In [4]:
def insert(word: str) -> set:
    alphabet = 'abcdefghijklmnñopqrstuvwxyz'
    words_created = set()
    for i in range(len(word) + 1):
        for char in alphabet:
            new_word = word[:i] + char + word[i:]
            words_created.add(new_word)
    
    return words_created


def delete(word:str) -> set:
    words_created = set()
    for i in range(len(word)):
        new_word = word[:i] + word[i+1:]
        words_created.add(new_word)

    return words_created
    
def replace(word:str) ->set:
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    words_created = set()
    for i in range(len(word)): #hola
        for char in alphabet:
            new_word = word[:i] + char + word[i+1:]
            words_created.add(new_word)

    return words_created

def exchange(word:str)->set:
    words_created = set()
    for i in range(len(word) -1):
        new_word = word[:i] + word[i+1] + word[i] + word[i+2:]
        words_created.add(new_word)

    return words_created


**Funcion una edicion y dos ediciones**

Debemos recibir una palabra y aplicar las 4 operaciones basicas y devolver las palabras unicas a distancia uno o distancia dos:

In [5]:
def una_edicion(word:str, intercambiar: bool=False) ->set:
    set1 = insert(word)
    set2 = delete(word)
    set3 = replace(word)
    if not intercambiar:
        words_one_edition = set1.union(set2, set3)
    else:
        set4 = exchange(word)
        words_one_edition = set1.union(set2, set3, set4)

    return words_one_edition


def dos_ediciones(word:str,intercambiar:bool =False) ->set:
    palabras_ed1 = una_edicion(word, intercambiar)
    palabras_ed2 = set()
    for palabra in palabras_ed1:
        palabras_ed2 = palabras_ed2.union(una_edicion(palabra, intercambiar))
    return palabras_ed2